# RAPTOR Tree Building Notebook

Notebook này triển khai riêng bước **Embedding → UMAP → GMM clustering → summarization → xây cây RAPTOR** từ các chunk đã tiền xử lý.

Đầu vào dự kiến là file `chunks.jsonl` sinh ra từ `pipeline.py`.

## Các bước trong notebook

1. Load `chunks.jsonl`
2. Mã hóa mỗi chunk bằng **SBERT: `multi-qa-mpnet-base-cos-v1`**
3. Giảm chiều bằng **UMAP**
4. Chọn số cluster tối ưu bằng **BIC**
5. Gom nhóm bằng **GMM**
6. Tóm tắt từng cluster bằng OpenAI API nếu có `OPENAI_API_KEY`, hoặc fallback local
7. Lặp đệ quy để tạo cây nhiều tầng
8. Lưu các kết quả trung gian để xem từng giai đoạn

In [ ]:
# Nếu cần, bạn có thể cài thư viện trong notebook của mình:
# !pip install -q sentence-transformers umap-learn scikit-learn pandas matplotlib numpy tqdm tiktoken openai

In [2]:
! pip install umap-learn

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.8 MB 5.9 MB/s eta 0:00:01
   ----------- ---------------------------- 0.8/2.8 MB 3.5 MB/s eta 0:00:01
   ----------- ---------------------------- 0.8/2.8 MB 3.5 MB/s eta 0:00:01
   ----------- ---------------------------- 0.8/2.8 MB 3.5 MB/s eta 0:00:01
   ----------- ---------------------------- 0.8/2.8 MB 3.5 MB/s eta 0:00:01
   --------------- ------------------------ 1.0/2.8 MB 671.3 kB/s eta 0:00:03
   ------------------- -------------------- 1.3/2.8 MB 763.7 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 872.8 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.8 MB 872.8 kB/s eta 0:00:02
   ------------------------------ --------- 2.1/2.8 MB 938.3 kB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 938.3 kB/s eta 0:00:01
   ------------------------------ --------- 2.1/2.8 MB 938.3 kB/s eta 0:00:01
   --


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from __future__ import annotations

import json
import math
import os
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.mixture import GaussianMixture
from sentence_transformers import SentenceTransformer
import umap

In [ ]:
# =========================
# Config
# =========================

CHUNKS_PATH = Path("../data/processed/preprocess_full_no_assets")  # sửa nếu cần
OUTPUT_DIR = Path("notebook_outputs/raptor_tree")

EMBED_MODEL_NAME = "sentence-transformers/multi-qa-mpnet-base-cos-v1"
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.0
UMAP_N_COMPONENTS = 5

BIC_MAX_K = 8
MIN_CLUSTER_SIZE = 3
MIN_TOTAL_WORDS_TO_SPLIT = 1200
MAX_DEPTH = 4

OPENAI_MODEL = "gpt-3.5-turbo"
USE_OPENAI = bool(os.getenv(""))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR.resolve()}")
print(f"Using OpenAI API: {USE_OPENAI}")

In [ ]:
def find_chunks_jsonl(root: Path) -> List[Path]:
    if root.is_file() and root.name.endswith(".jsonl"):
        return [root]
    if root.is_dir():
        return sorted(root.rglob("chunks.jsonl"))
    return []

def load_chunks(path_or_dir: Path) -> pd.DataFrame:
    paths = find_chunks_jsonl(path_or_dir)
    if not paths:
        raise FileNotFoundError(
            f"Không tìm thấy chunks.jsonl tại: {path_or_dir}. "
            "Hãy trỏ CHUNKS_PATH tới thư mục output của pipeline.py hoặc file chunks.jsonl."
        )
    rows = []
    for p in paths:
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                item["_source_jsonl"] = str(p)
                rows.append(item)
    return pd.DataFrame(rows)

def get_text_for_node(row: pd.Series) -> str:
    return str(row.get("content", "")).strip()

def count_words(text: str) -> int:
    return len(re.findall(r"\S+", text))

def estimate_tokens(text: str) -> int:
    return max(1, math.ceil(count_words(text) * 1.3))

def preview_text(text: str, n_words: int = 80) -> str:
    words = re.findall(r"\S+", text)
    return " ".join(words[:n_words]) + (" ..." if len(words) > n_words else "")

In [ ]:
df = load_chunks(CHUNKS_PATH)
print("Loaded chunks:", len(df))
display(df.head(3))
print("Columns:", df.columns.tolist())

In [ ]:
text_df = df[df["chunk_type"].eq("text")].copy().reset_index(drop=True)
print("Text chunks:", len(text_df))
display(text_df[["doc_name", "page_number", "chunk_type", "chunk_index", "content"]].head(5))

In [ ]:
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL_NAME)

texts = [get_text_for_node(row) for _, row in text_df.iterrows()]
embeddings = embedder.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

print("Embeddings shape:", embeddings.shape)
np.save(OUTPUT_DIR / "leaf_embeddings.npy", embeddings)
text_df["embedding_index"] = range(len(text_df))
print("Saved:", OUTPUT_DIR / "leaf_embeddings.npy")

In [ ]:
sample_vec = embeddings[0]
print("Vector dimension:", len(sample_vec))
print("First 10 values:", np.round(sample_vec[:10], 4))

In [ ]:
n_samples = len(embeddings)
umap_components = min(UMAP_N_COMPONENTS, max(2, n_samples - 1))

reducer = umap.UMAP(
    n_neighbors=min(UMAP_N_NEIGHBORS, max(2, n_samples - 1)),
    n_components=umap_components,
    min_dist=UMAP_MIN_DIST,
    metric="cosine",
    random_state=42,
)

reduced = reducer.fit_transform(embeddings)
print("Reduced shape:", reduced.shape)
np.save(OUTPUT_DIR / "leaf_umap.npy", reduced)

reduced_df = pd.DataFrame(reduced, columns=[f"umap_{i}" for i in range(reduced.shape[1])])
display(reduced_df.head())

In [ ]:
if reduced.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(reduced[:, 0], reduced[:, 1], s=12)
    ax.set_title("Leaf chunks after UMAP")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.show()
else:
    print("Không đủ chiều để vẽ 2D.")

In [ ]:
def bic_search(X: np.ndarray, max_k: int = 8) -> Tuple[int, pd.DataFrame]:
    records = []
    max_k = max(2, min(max_k, len(X)))
    for k in range(2, max_k + 1):
        gmm = GaussianMixture(
            n_components=k,
            covariance_type="full",
            random_state=42,
            reg_covar=1e-6,
            n_init=5,
        )
        gmm.fit(X)
        records.append({"k": k, "bic": gmm.bic(X), "aic": gmm.aic(X)})
    result = pd.DataFrame(records)
    best_k = int(result.sort_values("bic").iloc[0]["k"])
    return best_k, result

best_k, bic_df = bic_search(reduced, BIC_MAX_K)
print("Best k by BIC:", best_k)
display(bic_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bic_df["k"], bic_df["bic"], marker="o")
ax.set_title("GMM BIC search")
ax.set_xlabel("k")
ax.set_ylabel("BIC")
plt.show()

In [ ]:
gmm = GaussianMixture(
    n_components=best_k,
    covariance_type="full",
    random_state=42,
    reg_covar=1e-6,
    n_init=10,
)
cluster_labels = gmm.fit_predict(reduced)
text_df["cluster"] = cluster_labels

cluster_counts = text_df["cluster"].value_counts().sort_index().reset_index()
cluster_counts.columns = ["cluster", "count"]
display(cluster_counts)

if reduced.shape[1] >= 2:
    fig, ax = plt.subplots(figsize=(8, 6))
    scatter = ax.scatter(reduced[:, 0], reduced[:, 1], c=cluster_labels, s=14, cmap="tab10")
    ax.set_title("Leaf chunks clustered by GMM")
    ax.set_xlabel("UMAP-1")
    ax.set_ylabel("UMAP-2")
    plt.colorbar(scatter, ax=ax, label="cluster")
    plt.show()

In [ ]:
def sentence_split(text: str) -> List[str]:
    parts = re.split(r"(?<=[.!?។])\s+", text.strip())
    return [p.strip() for p in parts if p.strip()]

def local_extractive_summary(texts: List[str], max_sentences: int = 4) -> str:
    joined = " ".join(texts)
    sentences = sentence_split(joined)
    if not sentences:
        return joined[:1200]

    words = re.findall(r"[\wÀ-Ỵà-ỵĐđ]+", joined.lower())
    freq: Dict[str, int] = {}
    for w in words:
        if len(w) <= 2:
            continue
        freq[w] = freq.get(w, 0) + 1

    scored = []
    for idx, sent in enumerate(sentences):
        sent_words = re.findall(r"[\wÀ-Ỵà-ỵĐđ]+", sent.lower())
        score = sum(freq.get(w, 0) for w in sent_words)
        scored.append((score, idx, sent))

    top = sorted(scored, reverse=True)[:max_sentences]
    top = sorted(top, key=lambda x: x[1])
    summary = " ".join(sent for _, _, sent in top)
    return summary[:1500]

def openai_summary(texts: List[str], model: str = OPENAI_MODEL) -> str:
    from openai import OpenAI
    client = OpenAI()
    prompt = (
        "Tóm tắt các mục quan trọng trong đoạn văn sau, giữ lại tiêu chí lâm sàng, "
        "xét nghiệm, ngưỡng giá trị, và ý nghĩa chẩn đoán nếu có. Viết ngắn gọn, rõ ràng.\n\n"
        + "\n\n".join(f"- {t}" for t in texts)
    )
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "Bạn là trợ lý tóm tắt tài liệu y khoa."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )
    return resp.choices[0].message.content.strip()

def summarize_cluster(texts: List[str]) -> str:
    if USE_OPENAI:
        try:
            return openai_summary(texts)
        except Exception as e:
            print("OpenAI summarization failed, fallback to local extractive summarizer:", e)
    return local_extractive_summary(texts)

def total_cluster_words(texts: List[str]) -> int:
    return sum(count_words(t) for t in texts)

In [ ]:
def build_next_layer(nodes_df: pd.DataFrame, depth: int):
    texts = nodes_df["text"].tolist()
    emb = embedder.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    n_samples = len(emb)
    if n_samples < MIN_CLUSTER_SIZE:
        return pd.DataFrame(), nodes_df, emb, pd.DataFrame()

    n_neighbors = min(UMAP_N_NEIGHBORS, max(2, n_samples - 1))
    n_components = min(UMAP_N_COMPONENTS, max(2, n_samples - 1))
    reducer = umap.UMAP(
        n_neighbors=n_neighbors,
        n_components=n_components,
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        random_state=42,
    )
    red = reducer.fit_transform(emb)

    best_k, bic_df = bic_search(red, max_k=min(BIC_MAX_K, n_samples))
    if best_k <= 1:
        return pd.DataFrame(), nodes_df, red, bic_df

    gmm = GaussianMixture(
        n_components=best_k,
        covariance_type="full",
        random_state=42,
        reg_covar=1e-6,
        n_init=10,
    )
    labels = gmm.fit_predict(red)

    work = nodes_df.copy().reset_index(drop=True)
    work["cluster"] = labels
    work["depth"] = depth

    parent_rows = []
    for cluster_id in sorted(work["cluster"].unique()):
        cluster_df = work[work["cluster"] == cluster_id].copy()
        cluster_texts = cluster_df["text"].tolist()
        summary_text = summarize_cluster(cluster_texts)
        parent_rows.append({
            "node_id": f"d{depth}_c{cluster_id}",
            "depth": depth,
            "cluster_id": int(cluster_id),
            "n_children": len(cluster_df),
            "child_node_ids": cluster_df["node_id"].tolist(),
            "text": summary_text,
            "source_pages": sorted(set(cluster_df["page_number"].tolist())) if "page_number" in cluster_df.columns else [],
            "source_docs": sorted(set(cluster_df["doc_name"].tolist())) if "doc_name" in cluster_df.columns else [],
            "token_estimate": estimate_tokens(summary_text),
            "raw_texts": cluster_texts,
        })

    parent_df = pd.DataFrame(parent_rows)
    parent_df["cluster_words"] = parent_df["raw_texts"].apply(total_cluster_words)
    parent_df["bic_best_k"] = best_k
    parent_df["bic_table"] = [bic_df.to_dict(orient="records")] * len(parent_df)

    return parent_df, work, red, bic_df

print("Defined build_next_layer().")

In [ ]:
@dataclass
class RaptorNode:
    node_id: str
    depth: int
    text: str
    source_docs: List[str] = field(default_factory=list)
    source_pages: List[int] = field(default_factory=list)
    child_node_ids: List[str] = field(default_factory=list)
    cluster_id: Optional[int] = None
    token_estimate: int = 0
    is_leaf: bool = False

def build_raptor_tree(leaf_df: pd.DataFrame, max_depth: int = MAX_DEPTH) -> Dict[str, Any]:
    nodes_by_level: List[pd.DataFrame] = []
    current = leaf_df.copy().reset_index(drop=True)

    current = current.rename(columns={"content": "text"})
    current["node_id"] = [f"leaf_{i:06d}" for i in range(len(current))]
    current["depth"] = 0
    current["token_estimate"] = current["text"].apply(estimate_tokens)
    current["is_leaf"] = True
    nodes_by_level.append(current)

    layer_summaries: List[Dict[str, Any]] = []
    all_nodes: Dict[str, Dict[str, Any]] = {}

    for _, row in current.iterrows():
        all_nodes[row["node_id"]] = {
            "node_id": row["node_id"],
            "depth": int(row["depth"]),
            "text": row["text"],
            "source_docs": [row.get("doc_name")] if row.get("doc_name") else [],
            "source_pages": [int(row.get("page_number"))] if pd.notna(row.get("page_number")) else [],
            "child_node_ids": [],
            "cluster_id": None,
            "token_estimate": int(row["token_estimate"]),
            "is_leaf": True,
        }

    depth = 1
    while depth <= max_depth:
        layer_input = nodes_by_level[-1]
        total_words = sum(count_words(t) for t in layer_input["text"].tolist())

        if len(layer_input) < MIN_CLUSTER_SIZE or total_words < MIN_TOTAL_WORDS_TO_SPLIT:
            layer_summaries.append({
                "depth": depth,
                "status": "stop",
                "reason": "too_few_nodes_or_too_small",
                "nodes": len(layer_input),
                "total_words": int(total_words),
            })
            break

        next_layer_df, clustered_work, red, bic_df = build_next_layer(layer_input, depth)
        if next_layer_df is None or len(next_layer_df) == 0:
            layer_summaries.append({
                "depth": depth,
                "status": "stop",
                "reason": "no_next_layer",
                "nodes": len(layer_input),
                "total_words": int(total_words),
            })
            break

        nodes_by_level.append(next_layer_df)

        for _, row in next_layer_df.iterrows():
            all_nodes[row["node_id"]] = {
                "node_id": row["node_id"],
                "depth": int(row["depth"]),
                "text": row["text"],
                "source_docs": row["source_docs"],
                "source_pages": row["source_pages"],
                "child_node_ids": row["child_node_ids"],
                "cluster_id": int(row["cluster_id"]),
                "token_estimate": int(row["token_estimate"]),
                "is_leaf": False,
            }

        layer_summaries.append({
            "depth": depth,
            "status": "ok",
            "input_nodes": len(layer_input),
            "output_nodes": len(next_layer_df),
            "best_k": int(next_layer_df["bic_best_k"].iloc[0]),
            "bic": bic_df.to_dict(orient="records"),
            "cluster_sizes": next_layer_df["n_children"].tolist(),
        })

        if len(next_layer_df) <= 1:
            break

        current = next_layer_df[["node_id", "text", "depth", "source_docs", "source_pages", "child_node_ids", "cluster_id", "token_estimate"]].copy()
        depth += 1

    return {
        "model_name": EMBED_MODEL_NAME,
        "use_openai": USE_OPENAI,
        "levels": len(nodes_by_level),
        "layer_summaries": layer_summaries,
        "all_nodes": all_nodes,
        "nodes_by_level": [lvl.to_dict(orient="records") for lvl in nodes_by_level],
    }

print("Defined build_raptor_tree().")

In [ ]:
target_doc = text_df["doc_name"].iloc[0]
doc_df = text_df[text_df["doc_name"] == target_doc].copy().reset_index(drop=True)

print("Target doc:", target_doc)
print("Leaf chunks:", len(doc_df))
display(doc_df[["page_number", "chunk_index", "content"]].head(8))

tree = build_raptor_tree(doc_df, max_depth=MAX_DEPTH)

print("Tree levels:", tree["levels"])
display(pd.DataFrame(tree["layer_summaries"]))

In [ ]:
for level_idx, level_records in enumerate(tree["nodes_by_level"]):
    level_df = pd.DataFrame(level_records)
    print("=" * 80)
    print(f"LEVEL {level_idx} | nodes = {len(level_df)}")
    if "text" in level_df.columns:
        show_cols = [c for c in ["node_id", "cluster_id", "n_children", "token_estimate", "source_pages", "text"] if c in level_df.columns]
        show_df = level_df[show_cols].copy()
        if "text" in show_df.columns:
            show_df["preview"] = show_df["text"].astype(str).apply(preview_text)
            show_df = show_df.drop(columns=["text"])
        display(show_df.head(10))
    else:
        display(level_df.head(10))

In [ ]:
tree_path = OUTPUT_DIR / "raptor_tree.json"
with tree_path.open("w", encoding="utf-8") as f:
    json.dump(tree, f, ensure_ascii=False, indent=2)

leaf_out = pd.DataFrame(tree["nodes_by_level"][0])
leaf_out.to_csv(OUTPUT_DIR / "leaf_nodes.csv", index=False, encoding="utf-8-sig")

print("Saved tree:", tree_path)
print("Saved leaf nodes:", OUTPUT_DIR / "leaf_nodes.csv")

In [ ]:
if len(tree["nodes_by_level"]) > 1:
    level_1 = pd.DataFrame(tree["nodes_by_level"][1])
    cols = [c for c in ["node_id", "cluster_id", "n_children", "text"] if c in level_1.columns]
    display(level_1[cols].head(10))
else:
    print("Tree chỉ có 1 level; chưa tạo được summary layer.")

## Gợi ý chạy

- Chạy toàn bộ notebook từ trên xuống.
- Nếu muốn build cây trên **nhiều file guideline**, hãy đổi `CHUNKS_PATH` thành thư mục chứa nhiều `chunks.jsonl`.
- Nếu có `OPENAI_API_KEY`, notebook sẽ dùng GPT để summarization.
- Nếu không có API key, notebook tự fallback sang summarizer nội bộ.
- Có thể tinh chỉnh:
  - `MIN_TOTAL_WORDS_TO_SPLIT`
  - `MIN_CLUSTER_SIZE`
  - `BIC_MAX_K`
  - `MAX_DEPTH`
  - `UMAP_N_NEIGHBORS`